Transformer Encoder-Decoder
====


  - encoder self-attention: bidirectional, no causal mask
  - decoder self-attention: causal mask
  - decoder cross-attention: queries from decoder, keys/values from encoder
  - feed-forward network after attention
  - stacked encoder and decoder blocks

https://github.com/huggingface/transformers/blob/main/src/transformers/models/bart/modeling_bart.py

In [1]:
import torch
import torch.nn as nn
import tiktoken

In [2]:
embedding_dim = 768
context_length = 1024
num_heads = 12

tokenizer = tiktoken.get_encoding("gpt2")
n_vocab = tokenizer.n_vocab

print(f"Vocabulary size: {n_vocab}")

Vocabulary size: 50257


### 1. Prepare the input

In [3]:
input_texts = ["Hello, how are you?", "Transformers are amazing!"]
input_ids = [tokenizer.encode(t) for t in input_texts]
print(f"input_ids: {input_ids}")

# apply padding to input_ids to make them the same length
pad_token_id = tokenizer.eot_token  # GPT-2 tokenizer has no dedicated pad token
max_len = max(len(ids) for ids in input_ids)

padded_input_ids = [
    ids + [pad_token_id] * (max_len - len(ids))
    for ids in input_ids
]

input_ids_tensor = torch.tensor(padded_input_ids, dtype=torch.long)

# length-based masking for attention
lengths = torch.tensor([len(ids) for ids in input_ids])
positions = torch.arange(max_len).unsqueeze(0)
attention_mask = positions >= lengths.unsqueeze(1)  # True means padding

print("input_ids_tensor:", input_ids_tensor.shape)  # [batch_size, seq_len]
print("input_ids_tensor:\n", input_ids_tensor)
print("attention_mask:", attention_mask.shape)
print("attention_mask:\n", attention_mask)

input_ids: [[15496, 11, 703, 389, 345, 30], [41762, 364, 389, 4998, 0]]
input_ids_tensor: torch.Size([2, 6])
input_ids_tensor:
 tensor([[15496,    11,   703,   389,   345,    30],
        [41762,   364,   389,  4998,     0, 50256]])
attention_mask: torch.Size([2, 6])
attention_mask:
 tensor([[False, False, False, False, False, False],
        [False, False, False, False, False,  True]])


In [4]:
tok_embeddings = nn.Embedding(
    num_embeddings=n_vocab, embedding_dim=embedding_dim
)

class SinusoidalPositionEmbedding(nn.Module):
    def __init__(self, context_length, embedding_dim):
        super().__init__()
        position = torch.arange(context_length).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, embedding_dim, 2) * (-torch.log(torch.tensor(10000.0)) / embedding_dim)
        )

        pe = torch.zeros(context_length, embedding_dim)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe)

    def forward(self, positions):
        return self.pe[positions]


pos_embeddings = SinusoidalPositionEmbedding(
    context_length=context_length, embedding_dim=embedding_dim
)

tok_emb = tok_embeddings(input_ids_tensor)

batch_size, seq_len = input_ids_tensor.shape

positions = torch.arange(seq_len, dtype=torch.long, device=input_ids_tensor.device)
pos_emb = pos_embeddings(positions)  # [seq_len, embedding_dim]
print(f"pos_emb.shape: {pos_emb.shape}")  # [seq_len, 768]

batch_x = tok_emb + pos_emb

print(f"Input x: {batch_x.shape}")

pos_emb.shape: torch.Size([6, 768])
Input x: torch.Size([2, 6, 768])
